# Step 5 — BabelBrain: Acoustic & Thermal Simulation

Runs BabelBrain in batch mode (no GUI) using the trajectory file from Step 4 (PlanTUS).
Executes three stages per subject:
- **5a** Domain generation (tissue mask from SimNIBS mesh)
- **5b** Acoustic simulation (FDTD, GPU)
- **5c** Thermal simulation (Bio-heat equation)

**Prerequisites:**
- Step 1: `m2m_{sub_id}/` exists
- Step 4: `*_brainsight.txt` exists (PlanTUS output)

[← Step 4 — PlanTUS](step04_planTUS.ipynb) &nbsp;|&nbsp; **Step 5**


In [ ]:
# Reference card
# %run z_references/step5_reference.py

In [ ]:
# -----------------------------------------------------------------------
# Settings — edit here before running
# -----------------------------------------------------------------------

SITE_YAML   = "../config/sites/site_RIKEN_AK.yaml"
SUB_ID      = "sub-NS"

# ── Target (must match Step 4 output) ────────────────────────────────
TARGET_NAME = "aMCC_NeuroSynthTopic112"
TARGET_SIDE = "_R"                  # "_R", "_L", or ""

# ── Transducer ───────────────────────────────────────────────────────
# Leave as None to auto-load from config/transducers/{transducer}.yaml
# Set explicitly to override (e.g. for a different transducer or freq).
TX_SYSTEM   = None   # e.g. "CTX_500"      → babelbrain_id in transducer YAML
FREQUENCY   = None   # Hz, e.g. 500e3      → frequency_kHz * 1e3
APERTURE    = None   # m,  e.g. 64e-3      → active_diameter_mm / 1e3
FOCAL_LENGTH= None   # m,  e.g. 63.2e-3   → radius_of_curvature_mm / 1e3
Z_BEYOND    = 40e-3                 # m (simulation depth beyond target)

# ── Simulation quality ────────────────────────────────────────────────
PPW         = 6                     # points per wavelength (6=fast, 9=converged)

# ── Computing device ─────────────────────────────────────────────────
# Local Mac (Metal):  COMPUTING_DEVICE = 'M1', COMPUTING_BACKEND = 3
# Oscar (CUDA):       COMPUTING_DEVICE = 'A100', COMPUTING_BACKEND = 1
COMPUTING_DEVICE  = "M1"            # device name label
COMPUTING_BACKEND = 3               # 1=CUDA, 2=OpenCL, 3=Metal

# ── CT input ─────────────────────────────────────────────────────────
USE_CT      = False                 # True if CT scan available
CT_PATH     = ""                    # path to CT NIfTI (if USE_CT=True)
CT_TYPE     = 1                     # 1=real CT, 2=ZTE, 3=PETRA

# ── Thermal profile ───────────────────────────────────────────────────
BASE_ISPPA  = 5.0                   # W/cm² (reference intensity for BHTE)
THERMAL_PROFILE = [
    {"DC": 0.3, "PRF": 10.0, "Duration": 40.0, "DurationOff": 40.0, "Repetitions": 1},
]

# ── Run options ───────────────────────────────────────────────────────
DRY_RUN     = False                 # True = validate paths only
REUSE_FILES = True                  # True = skip if output already exists

print("Settings loaded.")
print(f"  Subject:  {SUB_ID}")
print(f"  Target:   {TARGET_NAME}{TARGET_SIDE}")
print(f"  Tx:       {TX_SYSTEM or '(auto)'}  freq={(str(int(FREQUENCY/1e3))+'kHz') if FREQUENCY else '(auto)'}  PPW={PPW}")
print(f"  Device:   {COMPUTING_DEVICE} (backend={COMPUTING_BACKEND})")
print(f"  DRY_RUN:  {DRY_RUN}")


In [ ]:
# -----------------------------------------------------------------------
# Imports & path setup
# -----------------------------------------------------------------------
import sys
import os
import time
from pathlib import Path
from multiprocessing import Process, Queue

import numpy as np

# ── Make src/ importable ──────────────────────────────────────────────
SRC_DIR = str((Path("../src")).resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

SITE_YAML = str(Path(SITE_YAML).resolve())

# ── BabelBrain imports ────────────────────────────────────────────────
from BabelBrain.CalculateMaskProcess import CalculateMaskProcess
from BabelBrain.CalculateFieldProcess import CalculateFieldProcess
from BabelBrain.Babel_Thermal.CalculateThermalProcess import CalculateThermalProcess
from ThermalModeling.CalculateTemperatureEffects import GetThermalOutName
from TranscranialModeling.BabelIntegrationBASE import GetSmallestSOS

# ── Pipeline utils ────────────────────────────────────────────────────
from utils import (
    load_site_config,
    resolve_data_dir,
    normalise_sub_id,
    resolve_sub_dir,
)

print("Imports complete.")


In [ ]:
# -----------------------------------------------------------------------
# Step 5-0 — Resolve paths and transducer parameters
# -----------------------------------------------------------------------
cfg = load_site_config(SITE_YAML)
data_dir = resolve_data_dir(cfg)
sub_id_full, sub_id_bare = normalise_sub_id(SUB_ID)

sub_dir = resolve_sub_dir(data_dir, sub_id_bare, sub_id_full)
m2m_dir = sub_dir / f"m2m_{sub_id_full}"
T1W     = str(m2m_dir / "T1.nii.gz")

# ── Resolve transducer parameters from YAML (if not overridden in Settings) ──
tx_cfg = cfg.get("transducer_cfg", {})
if tx_cfg:
    if TX_SYSTEM   is None: TX_SYSTEM    = tx_cfg.get("babelbrain_id")
    if FREQUENCY   is None: FREQUENCY    = tx_cfg.get("frequency_kHz", 0) * 1e3
    if APERTURE    is None: APERTURE     = tx_cfg.get("active_diameter_mm", 0) / 1e3
    if FOCAL_LENGTH is None: FOCAL_LENGTH = tx_cfg.get("radius_of_curvature_mm", 0) / 1e3
else:
    print("NOTE: transducer_cfg not found in site config; using Settings values.")

missing_tx = [n for n, v in [("TX_SYSTEM", TX_SYSTEM), ("FREQUENCY", FREQUENCY),
                               ("APERTURE", APERTURE), ("FOCAL_LENGTH", FOCAL_LENGTH)] if v is None]
if missing_tx:
    raise ValueError(f"Transducer parameters not resolved: {missing_tx}. Set explicitly in Settings.")

# Trajectory file from Step 4 (PlanTUS output)
plantus_target_folder = (
    m2m_dir / "PlanTUS"
    / f"{sub_id_full}_T1w_{TARGET_NAME}_mask_native{TARGET_SIDE}"
)
trajectory_file = str(
    plantus_target_folder
    / f"{sub_id_full}_T1w_{TARGET_NAME}{TARGET_SIDE}_target_coordinates_Brainsight_PlanTUS.txt"
)

# Validate
assert m2m_dir.exists(), f"m2m directory not found: {m2m_dir}"
assert Path(T1W).exists(), f"T1.nii.gz not found: {T1W}"
assert Path(trajectory_file).exists(), f"Trajectory file not found: {trajectory_file}"
if USE_CT:
    assert Path(CT_PATH).exists(), f"CT file not found: {CT_PATH}"

print(f"Subject:      {sub_id_full}")
print(f"m2m_dir:      {m2m_dir}")
print(f"T1W:          {T1W}")
print(f"Trajectory:   {trajectory_file}")
print(f"Transducer:   {TX_SYSTEM}  {int(FREQUENCY/1e3)} kHz  ∅{APERTURE*1e3:.1f}mm  f={FOCAL_LENGTH*1e3:.1f}mm")
print(f"Use CT:       {USE_CT}")


In [ ]:
# -----------------------------------------------------------------------
# Step 5a — Domain generation
# Generates tissue mask from SimNIBS mesh + trajectory.
# Output: m2m_dir/{ID}_{TX}_{freq}kHz_{PPW}PPW_BabelViscoInput.nii.gz
# -----------------------------------------------------------------------
ID = f"{sub_id_full}_T1w_{TARGET_NAME}{TARGET_SIDE}"

domain_file = os.path.join(
    str(m2m_dir),
    f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW_BabelViscoInput.nii.gz"
)

if REUSE_FILES and os.path.isfile(domain_file):
    print(f"[5a] Skipping — domain file already exists: {Path(domain_file).name}")
elif DRY_RUN:
    print(f"[5a] DRY RUN — would generate: {Path(domain_file).name}")
else:
    print("[5a] Domain generation started...")
    t0 = time.time()

    SmallestSOS = GetSmallestSOS(TX_SYSTEM)

    kargs = {}
    kargs['deviceName']        = COMPUTING_DEVICE
    kargs['COMPUTING_BACKEND'] = COMPUTING_BACKEND
    kargs['T1WIso']            = T1W
    kargs['simbnibs_path']     = str(m2m_dir)
    kargs['TrajFile']          = trajectory_file
    kargs['Frequency']         = FREQUENCY
    kargs['BasePPW']           = PPW
    kargs['TxSystem']          = TX_SYSTEM
    kargs['bUseCT']            = USE_CT
    kargs['CT_or_ZTE_input']   = CT_PATH if USE_CT else ''
    kargs['CTType']            = CT_TYPE
    kargs['bReuseFiles']       = REUSE_FILES
    kargs['SmallestSOS']       = SmallestSOS

    Q = Queue()
    p = Process(target=CalculateMaskProcess, args=(Q,), kwargs=kargs)
    p.start()
    p.join()
    result = Q.get()

    print(f"[5a] Done in {time.time()-t0:.1f}s")
    print(f"     Output: {Path(domain_file).name}")

In [ ]:
# -----------------------------------------------------------------------
# Step 5b — Acoustic simulation
# Output: m2m_dir/{ID}_{TX}_{freq}kHz_{PPW}PPW_Foc{f}_Diam{d}_DataForSim.h5
# -----------------------------------------------------------------------
acoustic_file = os.path.join(
    str(m2m_dir),
    f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW"
    f"_Foc{FOCAL_LENGTH*1e3:05.1f}_Diam{APERTURE*1e3:05.1f}_DataForSim.h5"
)

if REUSE_FILES and os.path.isfile(acoustic_file):
    print(f"[5b] Skipping — acoustic file already exists: {Path(acoustic_file).name}")
elif DRY_RUN:
    print(f"[5b] DRY RUN — would generate: {Path(acoustic_file).name}")
else:
    print("[5b] Acoustic simulation started...")
    t0 = time.time()

    kargs = {}
    kargs['deviceName']        = COMPUTING_DEVICE
    kargs['COMPUTING_BACKEND'] = COMPUTING_BACKEND
    kargs['T1WIso']            = T1W
    kargs['TxSystem']          = TX_SYSTEM
    kargs['Frequency']         = FREQUENCY
    kargs['PPW']               = PPW
    kargs['Aperture']          = APERTURE
    kargs['FocalLength']       = FOCAL_LENGTH
    kargs['zLengthBeyonFocalPointWhenNarrow'] = Z_BEYOND
    kargs['bUseCT']            = USE_CT
    kargs['ID']                = ID
    kargs['simbnibs_path']     = str(m2m_dir)

    Q = Queue()
    p = Process(target=CalculateFieldProcess, args=(Q,), kwargs=kargs)
    p.start()
    p.join()
    result = Q.get()

    print(f"[5b] Done in {time.time()-t0:.1f}s")
    print(f"     Output: {Path(acoustic_file).name}")

In [ ]:
# -----------------------------------------------------------------------
# Step 5c — Thermal simulation
# Solves Bio-heat Thermal Equation (BHTE) for all DC/PRF/Duration combos.
# Output: *_Isppa{isppa}_DC{dc}_PRF{prf}_Dur{dur}.h5 + CSV summary
# -----------------------------------------------------------------------
if not os.path.isfile(acoustic_file):
    print("[5c] Skipping — acoustic file not found. Run Step 5b first.")
else:
    # Check if all thermal outputs already exist
    existing = []
    for combo in THERMAL_PROFILE:
        thermal_name = GetThermalOutName(
            acoustic_file,
            combo['Duration'],
            combo['DurationOff'],
            combo['DC'],
            BASE_ISPPA,
            combo['PRF'],
            combo.get('Repetitions', 1)
        ) + '.h5'
        if os.path.isfile(thermal_name):
            existing.append(thermal_name)

    if REUSE_FILES and len(existing) == len(THERMAL_PROFILE):
        print(f"[5c] Skipping — all {len(THERMAL_PROFILE)} thermal output(s) already exist")
    elif DRY_RUN:
        print(f"[5c] DRY RUN — would run {len(THERMAL_PROFILE)} thermal simulation(s)")
    else:
        print(f"[5c] Thermal simulation started ({len(THERMAL_PROFILE)} combination(s))...")
        t0 = time.time()

        kargs = {}
        kargs['deviceName']          = COMPUTING_DEVICE
        kargs['COMPUTING_BACKEND']   = COMPUTING_BACKEND
        kargs['Isppa']               = BASE_ISPPA
        kargs['TxSystem']            = TX_SYSTEM
        kargs['AllDC_PRF_Duration']  = THERMAL_PROFILE
        kargs['InputFile']           = acoustic_file
        if TX_SYSTEM in ['CTX_500', 'Single', 'H246', 'BSonix']:
            kargs['sel_p'] = 'p_amp'
        else:
            kargs['sel_p'] = 'p_amp'

        Q = Queue()
        p = Process(target=CalculateThermalProcess, args=(Q,), kwargs=kargs)
        p.start()
        p.join()
        result = Q.get()

        print(f"[5c] Done in {time.time()-t0:.1f}s")

In [ ]:
# -----------------------------------------------------------------------
# Step 5-QC — Summary
# Lists all output files for this subject/target.
# -----------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"  Subject: {sub_id_full}")
print(f"  Target:  {TARGET_NAME}{TARGET_SIDE}")
print(f"  Tx:      {TX_SYSTEM}  {int(FREQUENCY/1e3)} kHz  PPW={PPW}")
print(f"{'='*60}")

patterns = [
    f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW_BabelViscoInput.nii.gz",
    f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW_Foc*.h5",
    f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW_Foc*Thermal*.h5",
]

from glob import glob
for pattern in patterns:
    files = sorted(glob(os.path.join(str(m2m_dir), pattern)))
    for f in files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"  [✓] {Path(f).name}  ({size_mb:.1f} MB)")